<a href="https://colab.research.google.com/github/matstabel/deep_learning/blob/main/ML_Exercise5_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment: Hyperparameter Tuning with Keras Tuner

In this assignment, you'll practice automated hyperparameter tuning using Keras Tuner by building and optimizing a simple neural network model.

### Objective
Create a neural network model using Keras Tuner to systematically search for optimal hyperparameters. Specifically, you will search over:

- **Number of neurons:** Search among reasonable numbers of neurons in the hidden layer.
- **Learning rate:** Search among learning rates of `1e-2`, `1e-3`, or `1e-4`.
- **Architecture:** Use exactly **one hidden layer**.
- **Optimizer:** Use the Adam optimizer.

### Instructions

1. **Data Preparation**: Use Fashion MNIST.

2. **Define the Model**: Use the following template for your `build_model` function:

```python
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice(name="units", values=["YOUR CODE HERE"])
    learning_rate = hp.Choice(name="learning_rate", values=[1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    return model
```

3. **Hyperparameter Search**: Use Keras Tuner (`BayesianOptimization`) to run the search for optimal hyperparameters. Set `keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` and examine what this does. Record the top 4 configurations.

4. **Optimal number of epochs**: Find the optimal number of epochs using the best training configuration.

5. **Train on the full dataset and test**

## Let's begin

In [ ]:
!pip install keras-tuner -q

**Explanation for the number of neurons**: YOUR TEXT HERE

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice("units", ["YOUR CODE HERE"])
    learning_rate = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
import kerastuner as kt

tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True,
)

In [ ]:
# 1. Load the Fashion MNIST dataset
import tensorflow as tf
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# 2. Preprocess the data:
"YOUR CODE HERE"

`keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` does what?

In [ ]:
# Perform search
tuner.search(
"YOUR CODE HERE"
)

In [ ]:
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)

In [ ]:
def get_best_epoch(hp):
    model = build_model(hp)

    history = model.fit(
      "YOUR CODE HERE")
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [ ]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        train_images, train_labels,
        batch_size=128, epochs="YOUR CODE HERE")
    return model

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(test_images, test_labels)
    best_models.append(model)

In [ ]:
for idx, model in enumerate(best_models):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    print(f"Model {idx+1}: Test Loss = {loss:.4f}, Test Accuracy = {acc:.4f}")


In [ ]:
import pandas as pd

results = []
for idx, (hp, model) in enumerate(zip(best_hps, best_models)):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    results.append({
        "Model": idx+1,
        "Units": hp.get("units"),
        "Learning Rate": hp.get("learning_rate"),
        "Test Accuracy": f"{acc*100:.2f}%",
        "Test Loss": f"{loss:.3f}"
    })

df_results = pd.DataFrame(results)
print(df_results)